# Hansen《Econometrics》第 2 章习题解答

**Chapter 2 Conditional Expectation and Projection**

对应书稿 PDF 第 80–81 页（印刷页 60–61），§2.34 Exercises。

完整推导与**面向初学者的详细注释**见同目录 `Hansen_Ch02_Exercises_Solutions.md`（强烈建议先读它的 §0、§1 概念铺垫）。本 notebook 只做 **Exercise 2.4、2.16** 的数值验证，并汇总其余题目答案。

> **写给只学过李子奈/陈强的同学：** 本章几乎不谈 OLS 估计量，而是在**总体**层面把"要估计的东西"讲清楚。记住两个计算技巧——**迭代期望**（"信息集小的赢"）与**条件期望定理**（"条件里的 $X$ 能提出来"），绝大多数证明题都用得上。


## Exercise 2.1–2.3（迭代期望与条件期望定理）

**核心技巧（反复用）：**
$$E[YX]\xRightarrow{\text{LIE}}E\big[E[YX\mid X]\big]\xRightarrow{X\text{ 提出来}}E\big[X\,E[Y\mid X]\big].$$

**2.1** 反复套迭代期望（"信息集小的赢"），逐层剥掉 $X_3$、再 $X_2$：
$$E[E[E[Y\mid X_1,X_2,X_3]\mid X_1,X_2]\mid X_1]=E[Y\mid X_1].$$

**2.2** 若 $E[Y\mid X]=a+bX$，把 $X$ 提出来再代入：
$$E[YX]=E[X\,E[Y\mid X]]=E[X(a+bX)]=aE[X]+bE[X^2].$$

**2.3** 证 Theorem 2.4.4（$E[e\mid X]=0\Rightarrow E[h(X)e]=0$）：
$$E[h(X)e]=E[E[h(X)e\mid X]]=E[h(X)\underbrace{E[e\mid X]}_{=0}]=0.$$
意义：零条件均值不仅消灭 $e$ 与 $X$ 的相关，也消灭 $e$ 与 $X$ **任何函数**（含 $X^2$）的相关——这是区分 2.10(True) 与 2.11(False) 的关键。


## Exercise 2.4（离散联合分布手算条件矩）

| | $X=0$ | $X=1$ |
|--|------:|------:|
| $Y=0$ | 0.1 | 0.2 |
| $Y=1$ | 0.4 | 0.3 |

**三步法：** ① 边际 $P(X)=$ 列和；② 条件 $P(Y\mid X=x)=P(Y,X=x)/P(X=x)$；③ 按定义算矩。

**关键观察：** $Y\in\{0,1\}\Rightarrow Y^2=Y$（$0^2=0,1^2=1$），故 $E[Y^2\mid X]=E[Y\mid X]$，条件方差即伯努利方差 $p(1-p)$。

- $P(X=0)=P(X=1)=0.5$；
- $E[Y\mid X=0]=0.4/0.5=0.8$，$\mathrm{var}=0.8\times0.2=0.16$；
- $E[Y\mid X=1]=0.3/0.5=0.6$，$\mathrm{var}=0.6\times0.4=0.24$。

两条件方差不同 → **异方差**雏形（呼应 2.8、2.22）。下方代码数值复核。


In [ ]:
import numpy as np
import pandas as pd

# joint P(Y,X): rows Y=0,1; cols X=0,1
P = np.array([[0.1, 0.2],
              [0.4, 0.3]])
assert np.isclose(P.sum(), 1.0)

px = P.sum(axis=0)
print("P(X) =", px)

rows = []
for x in [0, 1]:
    py_x = P[:, x] / px[x]
    ey = float(py_x[1])           # E[Y|X] since Y in {0,1}
    ey2 = ey                      # Y^2 = Y
    var = ey2 - ey**2
    rows.append({"X": x, "E[Y|X]": ey, "E[Y^2|X]": ey2, "Var[Y|X]": var})
    print(f"X={x}: E[Y|X]={ey:.4f}, E[Y^2|X]={ey2:.4f}, Var={var:.4f}")

pd.DataFrame(rows)


## Exercise 2.5–2.9 要点

- **2.5** 把被预测对象从 $Y$ 换成 $e^2$：其条件期望 $E[e^2\mid X]=\sigma^2(X)$ 由 Theorem 2.7（CEF 是最小均方误差预测）即为 $e^2$ 的最佳预测。
- **2.6** 先证 $\mathrm{cov}(m(X),e)=E[m(X)e]=0$（用 2.3），再得 $\mathrm{var}(Y)=\mathrm{var}(m(X))+\sigma^2$。即总体 $R^2=1-\sigma^2/\mathrm{var}(Y)$。
- **2.7** 条件版"方差 = 二阶矩 − 均值平方"：$\sigma^2(X)=E[Y^2\mid X]-(E[Y\mid X])^2$。
- **2.8** Poisson：$E[Y\mid X]=\mathrm{Var}[Y\mid X]=X'\beta$；CEF 恰好线性（满足零条件均值），但条件方差随 $X$ 变 → **异方差**（古典同方差不成立）。
- **2.9** 饱和模型：$2\times3=6$ 格需 6 参数。截距 $+X_1$ + 两个类别虚拟（留 $A$ 作基准、避免虚拟变量陷阱）+ 两个交互项。


## Exercise 2.10–2.14 True/False（本章灵魂：$E[e\mid X]=0$ 强于 $E[Xe]=0$）

| 题 | 答案 | 理由与反例 |
|:--:|:----:|------|
| 2.10 | **True** | $E[e\mid X]=0\Rightarrow E[X^2e]=0$（Thm 2.4.4，取 $h(x)=x^2$） |
| 2.11 | **False** | 反例：$X\sim N(0,1)$，$e=X^2-1$。$E[Xe]=E[X^3]-E[X]=0$；但 $E[X^2e]=E[X^4]-E[X^2]=3-1=2\ne0$ |
| 2.12 | **False** | 均值独立 $\ne$ 独立。反例 $e=Xu$，$X,u\stackrel{iid}{\sim}N(0,1)$：$E[e\mid X]=0$ 但 $e\mid X=x\sim N(0,x^2)$ 方差随 $x$ 变 |
| 2.13 | **False** | 投影正交推不出零条件均值。复用 2.11：$e=X^2-1$ 时 $E[Xe]=0$ 但 $E[e\mid X]=X^2-1\ne0$ |
| 2.14 | **False** | 同方差+均值独立仍不独立。反例：$X\in\{0,1\}$ 等概率，$e\mid X=0$ 取 $\pm1$（二点）、$e\mid X=1\sim N(0,1)$；两边均值 0、方差 1（同方差），但条件分布不同 → 不独立 |

**口诀：** 强（$E[e\mid X]=0$）$\Rightarrow$ 弱（$E[Xe]=0$）；均值独立 $\ne$ 独立；同方差 $\ne$ 独立。不要把"残差与 $X$ 不相关"（OLS 机械保证）当成"模型设定正确"。


## Exercise 2.15–2.16（BLP 与 CEF）

**2.15** 只有截距的 BLP：最小化 $E[(Y-\alpha)^2]$，FOC 得 $\alpha=E[Y]$（最佳常数预测 = 无条件均值，总体版"$\bar y$ 最小化平方偏差"）。

**2.16** $f(x,y)=\frac{3}{2}(x^2+y^2)$，$0\le x,y\le1$。

**BLP**（先积出内层 $\int_0^1(x^2+y^2)dy=x^2+\tfrac13$）：
- $E[X]=E[Y]=\tfrac58$，$E[X^2]=\tfrac{7}{15}$，$E[XY]=\tfrac38$；
- $\mathrm{var}(X)=\tfrac{73}{960}$，$\mathrm{cov}=-\tfrac{1}{64}$；
- $\boxed{\beta=-\tfrac{15}{73}\approx-0.205,\quad \alpha=\tfrac{55}{73}\approx0.753}$.

**CEF**：$f_X(x)=\tfrac32(x^2+\tfrac13)$，故
$$m(x)=E[Y\mid X=x]=\frac{\tfrac12 x^2+\tfrac14}{x^2+\tfrac13}.$$
$m(0)=0.75$、$m(1)=0.5625$，是有理函数（非线性）→ **BLP ≠ CEF**（BLP 只是对 CEF 的最佳线性近似）。下方代码数值复核。


In [ ]:
from scipy import integrate

def f(y, x):
    return 1.5 * (x**2 + y**2)

I, _ = integrate.dblquad(lambda y, x: f(y, x), 0, 1, 0, 1)
print("integral of density =", I)

EX, _ = integrate.dblquad(lambda y, x: x * f(y, x), 0, 1, 0, 1)
EY, _ = integrate.dblquad(lambda y, x: y * f(y, x), 0, 1, 0, 1)
EX2, _ = integrate.dblquad(lambda y, x: (x**2) * f(y, x), 0, 1, 0, 1)
EXY, _ = integrate.dblquad(lambda y, x: x * y * f(y, x), 0, 1, 0, 1)
print(f"E[X]={EX:.6f}, E[Y]={EY:.6f}, E[X^2]={EX2:.6f}, E[XY]={EXY:.6f}")

varX = EX2 - EX**2
cov = EXY - EX * EY
beta = cov / varX
alpha = EY - beta * EX
print(f"BLP: alpha={alpha:.6f}, beta={beta:.6f}")
print(f"exact: beta=-15/73={-15/73:.6f}, alpha=55/73={55/73:.6f}")

def m(x):
    fx = 1.5 * (x**2 + 1.0/3.0)
    num = 1.5 * (0.5 * x**2 + 0.25)
    return num / fx

for x in [0.0, 0.5, 1.0]:
    print(f"x={x}: m(x)={m(x):.6f}, BLP={alpha + beta*x:.6f}")

print("CEF is not affine in x => different from BLP")


## Exercise 2.17–2.22 要点

- **2.17** 两个矩方程解两个未知数（矩方法雏形）：第一分量 $\Rightarrow m=\mu$；第二分量 $\Rightarrow s=\sigma^2$。换成样本均值即样本均值/样本方差。
- **2.18** $X_3=\alpha_1+\alpha_2X_2$ ⇒ 存在非零 $c$ 使 $X'c=0$ a.s. ⇒ $Q_{XX}$ 奇异（**完全共线性**，总体版）；有效回归元为 $(1,X_2)$，原 $X$ 上系数不唯一。
- **2.19** $d(\beta)=E[(m(X)-X'\beta)^2]$ 的 FOC 得 $\beta=(E[XX'])^{-1}E[Xm(X)]$；再用 LIE 得 $E[Xm(X)]=E[XY]$。说明 BLP 既是"对 $Y$"也是"对 CEF"的最佳线性近似。
- **2.20** 有密度时 $E[1\{X\in\mathcal{X}\}Y]=\int_{\mathcal{X}}m(x)f_X(x)dx=E[1\{X\in\mathcal{X}\}m(X)]$，即 (2.57) 与 (2.6) 一致（实为 2.3 取 $h=1\{X\in\mathcal{X}\}$）。
- **2.21** 总体 OVB：$\gamma_1=\beta_1+(E[XX'])^{-1}E[XX_2]\beta_2$；故 $\gamma_1=\beta_1$ iff $E[XX_2]\beta_2=0$（漏掉变量无效应，或与 $X$ 不相关）。换成 $X_3$ 条件一般不同。
- **2.22** 剔除 $X_2$：新误差 $u=v'\beta_2+e$，$E[u^2\mid X_1]=\sigma^2+E[(v'\beta_2)^2\mid X_1]$ 一般依赖 $X_1$ ⇒ **诱发异方差**（除特殊结构外）。提醒：模型缩减不仅改变系数（OVB），还会破坏同方差 ⇒ 稳健标准误的动机之一。
